# V-Max BC pre-training on Colab (hanam/jeju, hard/easy pools)

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Data source**: the contest organizer's Drive folder, laid out as

```
Motion planning and prediction/train/
  hanam/<date>.tar.gz
  jeju/<date>.tar.gz
  livinglab/<date>.tar.gz     <- NOT used (the contest evaluates hanam/jeju)
```

Add that shared folder as a shortcut in your own Drive first (open the share -> "Add shortcut to Drive"), then fix `TRAIN_ROOT` below.

**What this notebook does**

1. `scripts/prepare_archives_91f.py` walks the per-date archives one at a time: extract -> convert to 91-step WOMD records (`make_91f`) -> score difficulty (`score_scenarios`) -> tar the result into a Drive cache -> delete the raw files. Peak local disk is one date, not the whole dataset, and a finished date is never converted twice (a killed session resumes by untarring the cache).
2. `scripts/split_hard_easy_pools.py --drop-frac 0.2 --hard-frac 0.4` throws away the bottom 20% (plain lane-keeping) per site and splits the rest into **4 pools**: `hanam_hard`, `hanam_easy`, `jeju_hard`, `jeju_easy`.
3. BC trains on all 4 as a weighted mixture, so the site ratio and the difficulty ratio are tuned independently.
4. Checkpoints go straight to Drive, and BC training resumes from the last one, so a disconnect mid-run costs one checkpoint interval.

Also upload the fixed 300-scenario **evaluation** set as a tar (so every model - local and Colab - is scored against the exact same set): locally, `tar -chf data/val_sample_shards_hanam.tar -C data/eval/val_sample_shards_hanam .` (~930MB, dereferenced so the symlinks survive), then upload it to `MyDrive/vmax_workdir/data/val_sample_shards_hanam.tar`.

Runtime > Change runtime type > pick a GPU. **Prefer an A100/L4 runtime if you have Colab Pro** - not for the GPU, for the vCPUs: the conversion step is pure CPU and a 2-vCPU T4 runtime converts roughly 6x slower.

In [ ]:
!nvidia-smi
!nproc && free -g && df -h /content

## 1. Mount Drive and check the layout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Edit if the shared folder landed somewhere else.
TRAIN_ROOT = "/content/drive/MyDrive/Motion planning and prediction/train"
DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
SITES = "hanam,jeju"  # livinglab deliberately excluded

os.environ["TRAIN_ROOT"] = TRAIN_ROOT
os.environ["DRIVE_WORKDIR"] = DRIVE_WORKDIR
os.environ["SITES"] = SITES

assert os.path.isdir(TRAIN_ROOT), f"Missing {TRAIN_ROOT} - fix the path (did you Add shortcut to Drive?)."
for site in SITES.split(","):
    sdir = os.path.join(TRAIN_ROOT, site)
    assert os.path.isdir(sdir), f"Missing {sdir}"
    archives = sorted(f for f in os.listdir(sdir) if f.endswith((".tar.gz", ".tgz", ".tar")))
    print(f"{site}: {len(archives)} date archives, e.g. {archives[:3]}")

# Only needed for the checkpoint sweep in section 9 - training itself never reads it.
EVAL_TAR = f"{DRIVE_WORKDIR}/data/val_sample_shards_hanam.tar"
HAS_EVAL_TAR = os.path.exists(EVAL_TAR)
print("eval set:", "found" if HAS_EVAL_TAR else f"MISSING {EVAL_TAR} - sections 6/9 will skip it")

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Smoke test: one date archive per site

Do not skip this. It validates the Drive path, the archive contents and the record schema in a few minutes, and - more importantly - it prints **seconds per file**, which is the number to multiply out before committing to the full run in step 4.

In [ ]:
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/smoke_91f \
    --sites "$SITES" \
    --scores-dir /content/data/smoke_scores \
    --max-archives-per-site 1
!find /content/data/smoke_91f -name '*.tfrecord' | wc -l
!du -sh /content/data/smoke_91f

In [ ]:
# Extrapolate before starting the full run: files/date x dates x s/file (from the smoke output above).
import os
n_files = sum(len(fs) for _, _, fs in os.walk("/content/data/smoke_91f"))
size_gb = sum(
    os.path.getsize(os.path.join(d, f))
    for d, _, fs in os.walk("/content/data/smoke_91f") for f in fs
) / 1e9
n_dates = sum(
    len([f for f in os.listdir(os.path.join(os.environ["TRAIN_ROOT"], s)) if f.endswith((".tar.gz", ".tgz", ".tar"))])
    for s in os.environ["SITES"].split(",")
)
n_smoke_dates = len(os.environ["SITES"].split(","))
print(f"{n_files} files / {n_smoke_dates} dates -> full run ~{n_files / n_smoke_dates * n_dates:,.0f} files, "
      f"~{size_gb / n_smoke_dates * n_dates:,.0f} GB of 91f output (Drive cache + local disk each need that much)")

## 4. Full conversion + scoring, cached to Drive

`--cache-dir` makes this restartable: each finished date is tarred to `MyDrive/vmax_workdir/cache_91f/<site>/<date>.tar`, and a re-run untars it instead of reconverting. When the session dies, re-run sections 1-2 and then this cell unchanged.

Two things to watch:

- **Drive quota**: the cache holds the entire 91-step dataset (the estimate printed above). If it does not fit, cap the run with `--max-archives-per-site N` - the pools built in step 5 will just cover fewer dates.
- **Wall clock**: conversion is CPU-bound and Colab gives you 2-12 vCPUs. If the extrapolation says more hours than a session allows, that is fine (the cache resumes), but a capped run gets you to training sooner.

In [ ]:
!rm -rf /content/data/smoke_91f
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/train_91f \
    --sites "$SITES" \
    --scores-dir "$DRIVE_WORKDIR/scores" \
    --cache-dir "$DRIVE_WORKDIR/cache_91f"
# --max-archives-per-site 8   # <- add this to cap the run (per site) if disk/time is short

## 5. Build the 4 training pools

`--drop-frac 0.2` discards the bottom 20% of each site outright (near-static lane keeping - the "직진 위주 단순 데이터 제거" step); `--hard-frac 0.4` then splits what remains into hard/easy **per site**, so `hanam_*` and `jeju_*` pools stay separately weightable.

In [ ]:
!rm -rf /content/data/shards/mixture_pools
!uv run python scripts/split_hard_easy_pools.py \
    "$DRIVE_WORKDIR/scores/combined_scores.csv" \
    /content/data/train_91f \
    /content/data/shards/mixture_pools \
    --sites "$SITES" --drop-frac 0.2 --hard-frac 0.4

## 6. Wire up checkpoints (Drive, persistent) and the fixed eval set

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs

if HAS_EVAL_TAR:
    !mkdir -p /content/data/eval/val_sample_shards_hanam
    !tar -xf "$EVAL_TAR" -C /content/data/eval/val_sample_shards_hanam
else:
    print("no eval tar - skipping (section 9 needs it)")

## 7. Train

Weights below bias toward the evaluated site (hanam) and toward hard scenarios: hanam 0.6 / jeju 0.4, hard 0.6 / easy 0.4. Nothing is dropped inside a pool - the weight only sets how often each pool is drawn from - so this is safe to retune between runs.

`total_timesteps=5_000_000` is roughly one pass over a ~59k-scenario pool at 80 steps. Bump it (e.g. `20_000_000`, the framework default scale) once TensorBoard shows `train/imitation_loss` still trending down at 5M.

**Memory**: 4 pools means 4 live tf.data pipelines. If the session OOMs, drop `algorithm.buffer_size` to 10000 first, then `num_envs` to 2 - or merge back to 2 pools with `scripts/merge_pools.py`.

**If the session disconnects mid-run**: re-run sections 1-2, then 4 (restores from the Drive cache), 5 and 6, then this cell unchanged - `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
import os

POOL_WEIGHTS = {"hanam_hard": 0.35, "hanam_easy": 0.25, "jeju_hard": 0.25, "jeju_easy": 0.15}
POOL_ROOT = "/content/data/shards/mixture_pools"

entries = []
for name, weight in POOL_WEIGHTS.items():
    pool_dir = os.path.join(POOL_ROOT, name)
    with open(os.path.join(pool_dir, "manifest.csv")) as fh:
        n = sum(1 for _ in fh) - 1  # minus header
    print(f"{name}: {n} shards, weight {weight}")
    entries.append(f"{{path: {pool_dir}/{name}.tfrecord@{n}, weight: {weight}}}")

MIXTURE = "[" + ", ".join(entries) + "]"
os.environ["MIXTURE"] = MIXTURE  # so the shell cell below sees it either way
print("\n" + MIXTURE)

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 8. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 9. After training: sweep checkpoints on the fixed held-out set

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_bc_run1 \
  --path_dataset /content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300 \
  --waymo_dataset true --batch_size 4